<a href="https://colab.research.google.com/github/ValentinaZubareva2906/make_AI_product/blob/main/prompting/2_2_resume_itog.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

In [2]:
!pip install langchain langchain-classic langchain-openai openai tiktoken -q

In [3]:
# Если используете ключ из курса, запустите эту ячейку
from langchain_openai import ChatOpenAI

course_api_key= "sk-CTWDfcT-MqN2gUhZh_3qbA"
#course_api_key = getpass(prompt='Введите API-ключ полученный в боте:')

# инициализируем языковую модель
llm = ChatOpenAI(api_key=course_api_key, model='gpt-4o-mini',
                 base_url="https://aleron-llm.neuraldeep.tech/")

In [4]:
import pandas as pd
from tqdm import tqdm

In [5]:
df = pd.read_csv('https://stepik.org/media/attachments/lesson/1110806/vacancies_messages_50.csv')

In [6]:
#df.head()

In [7]:
from langchain_classic import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.output_parsers import ResponseSchema
from langchain_classic.output_parsers import StructuredOutputParser

In [8]:
job_title_schema = ResponseSchema(
    name="job_title",
    description=(
        "Извлеки точное название вакансии, как указано в тексте, на том же языке. "
        "Игнорируй упоминания грейда (Senior, Middle и т. п.). "
        "Пример: 'Senior Python developer' → 'Python developer'. "
        "Если название вакансии не указано или его невозможно определить, верни None."
    )
)

company_schema = ResponseSchema(
    name="company",
    description=(
        "Извлеки название компании, как указано в тексте, на том же языке. "
        "Указывай только официальное название (без уточнений типа 'финтех', 'крупная компания' и т. п.). "
        "Если название компании не указано или его невозможно определить, верни None."
    )
)

salary_schema = ResponseSchema(
    name="salary",
    description=(
        "Извлеки информацию о зарплате в следующем формате: "
        "- Числа пиши без пробелов и сокращений (100к → 100000). "
        "- Указывай валюту: 'руб.' или '$' (после пробела). "
        "- Для диапазона: число-число валюта (без пробелов у тире), например: '100000-150000 руб.'. "
        "- Если только нижняя граница: 'от X валюта' (например, 'от 100000 руб.'). "
        "- Если только верхняя граница: 'до X валюта' (например, 'до 150000 руб.'). "
        "- Если зарплата почасовая: добавь 'в час' в конце (например, '500 руб. в час'). "
        "- Не включай слова: 'net', 'gross', 'на руки', 'премия', 'фикс', '% от продаж' и т. п. "
        "Если информация о зарплате отсутствует или её невозможно определить, верни None."
    )
)

tg_schema = ResponseSchema(
    name="tg",
    description=(
        "Извлеки контакты в Telegram в формате @username. "
        "Если несколько контактов — перечисли через запятую с пробелом (например, '@alice, @bob'). "
        "Если контакты отсутствуют или их невозможно определить, верни None."
    )
)

grade_schema = ResponseSchema(
    name="grade",
    description=(
        "Извлеки грейд из списка: intern, junior, junior+, middle, middle+, senior, lead. "
        "Если указано несколько грейдов — перечисли через запятую в порядке возрастания (например, 'junior, middle'). "
        "Если грейд не указан или его невозможно определить, верни None."
    )
)

response_schemas = [
    job_title_schema,
    company_schema,
    salary_schema,
    tg_schema,
    grade_schema
]

In [9]:
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = output_parser.get_format_instructions()

In [10]:
review_template = """\
Из следующего текста извлеки информацию в строго указанном формате.


Поля для извлечения:
- job_title: название вакансии.
- company: название компании.
- salary: информация о зарплате.
- tg: контакты в Telegram.
- grade: грейд.


Текст:
{text}

Инструкции по формату:
{format_instructions}
"""

In [11]:
prompt = ChatPromptTemplate.from_template(template=review_template)

In [12]:
dict_list = []

In [13]:
for text_input in tqdm(df['text']):
    try:
        # Формируем запрос
        messages = prompt.format_messages(
            text=text_input,
            format_instructions=format_instructions
        )

        # Отправляем запрос и получаем ответ
        response = llm.invoke(messages[0].content)

        # Парсим ответ
        parsed_output = output_parser.parse(response.content)

        dict_list.append(parsed_output)

    except:
        dict_list.append(None)  # При любой ошибке — None

100%|██████████| 50/50 [01:33<00:00,  1.88s/it]


In [14]:
print(len(dict_list))

50


In [17]:
print(dict_list)

[{'job_title': 'Python developer', 'company': 'Collectly', 'salary': '6000-9000 $', 'tg': '@ann_gfio', 'grade': 'senior'}, None, {'job_title': 'Database Administrator', 'company': 'Match Systems', 'salary': 'от 3000 $', 'tg': '@lex_kertis', 'grade': 'senior'}, None, None, {'job_title': 'DevOps инженер', 'company': 'Mad Devs', 'salary': 'до 5000 $', 'tg': '@recruiter_maddevs', 'grade': 'senior'}, {'job_title': 'Системный аналитик', 'company': 'Платформа', 'salary': '180000-300000 руб.', 'tg': '@Alexandrabogdanova_96', 'grade': 'middle+, senior, lead'}, None, None, None, None, {'job_title': 'QA automation Engineer (Python)', 'company': 'CFPS', 'salary': '180000-250000 руб.', 'tg': '@Ana_Itrecruiter', 'grade': 'middle, senior'}, None, {'job_title': 'Frontend разработчик', 'company': 'Travelata', 'salary': '280000-320000 руб.', 'tg': '@ann_gfio', 'grade': 'senior'}, None, None, None, {'job_title': 'Python developer', 'company': 'Skillbox', 'salary': '300000-400000 руб.', 'tg': '@ann_gfio',

In [15]:

DEFAULT_VALUES = {
    "job_title": "None",
    "company": "None",
    "salary": "None",
    "tg": "None",
    "grade": "None"
}

for item in dict_list:
    if isinstance(item, dict):
        # Берем значение из item, если нет — из DEFAULT_VALUES
        job_title = item.get("job_title", DEFAULT_VALUES["job_title"])
        company = item.get("company", DEFAULT_VALUES["company"])
        salary = item.get("salary", DEFAULT_VALUES["salary"])
        tg = item.get("tg", DEFAULT_VALUES["tg"])
        grade = item.get("grade", DEFAULT_VALUES["grade"])
        # ... и т. д.
    else:
        # Если item не словарь, берем всё из дефолтов
        job_title = DEFAULT_VALUES["job_title"]
        company = DEFAULT_VALUES["company"]
        salary = DEFAULT_VALUES["salary"]
        tg = DEFAULT_VALUES["tg"]
        grade = DEFAULT_VALUES["grade"]
        # ... и т. д.

In [22]:
for item in dict_list:
    job_title = item.get("job_title", "Не указано") if isinstance(item, dict) else "Не указано"
    company = item.get("company", "Не указано") if isinstance(item, dict) else "Не указано"
    salary = item.get("salary", "Не указано") if isinstance(item, dict) else "Не указано"
    tg = item.get("tg", "Не указано") if isinstance(item, dict) else "Не указано"
    grade = item.get("grade", "Не указано") if isinstance(item, dict) else "Не указано"


In [23]:
dict_list

[{'job_title': 'Python developer',
  'company': 'Collectly',
  'salary': '6000-9000 $',
  'tg': '@ann_gfio',
  'grade': 'senior'},
 None,
 {'job_title': 'Database Administrator',
  'company': 'Match Systems',
  'salary': 'от 3000 $',
  'tg': '@lex_kertis',
  'grade': 'senior'},
 None,
 None,
 {'job_title': 'DevOps инженер',
  'company': 'Mad Devs',
  'salary': 'до 5000 $',
  'tg': '@recruiter_maddevs',
  'grade': 'senior'},
 {'job_title': 'Системный аналитик',
  'company': 'Платформа',
  'salary': '180000-300000 руб.',
  'tg': '@Alexandrabogdanova_96',
  'grade': 'middle+, senior, lead'},
 None,
 None,
 None,
 None,
 {'job_title': 'QA automation Engineer (Python)',
  'company': 'CFPS',
  'salary': '180000-250000 руб.',
  'tg': '@Ana_Itrecruiter',
  'grade': 'middle, senior'},
 None,
 {'job_title': 'Frontend разработчик',
  'company': 'Travelata',
  'salary': '280000-320000 руб.',
  'tg': '@ann_gfio',
  'grade': 'senior'},
 None,
 None,
 None,
 {'job_title': 'Python developer',
  'compa

In [26]:
for i, item in enumerate(dict_list):
    if item is None:
        dict_list[i] = {
            "job_title": None,
            "company": None,
            "salary": None,
            "tg": None,
            "grade": None
        }


In [27]:
dict_list

[{'job_title': 'Python developer',
  'company': 'Collectly',
  'salary': '6000-9000 $',
  'tg': '@ann_gfio',
  'grade': 'senior'},
 {'job_title': None,
  'company': None,
  'salary': None,
  'tg': None,
  'grade': None},
 {'job_title': 'Database Administrator',
  'company': 'Match Systems',
  'salary': 'от 3000 $',
  'tg': '@lex_kertis',
  'grade': 'senior'},
 {'job_title': None,
  'company': None,
  'salary': None,
  'tg': None,
  'grade': None},
 {'job_title': None,
  'company': None,
  'salary': None,
  'tg': None,
  'grade': None},
 {'job_title': 'DevOps инженер',
  'company': 'Mad Devs',
  'salary': 'до 5000 $',
  'tg': '@recruiter_maddevs',
  'grade': 'senior'},
 {'job_title': 'Системный аналитик',
  'company': 'Платформа',
  'salary': '180000-300000 руб.',
  'tg': '@Alexandrabogdanova_96',
  'grade': 'middle+, senior, lead'},
 {'job_title': None,
  'company': None,
  'salary': None,
  'tg': None,
  'grade': None},
 {'job_title': None,
  'company': None,
  'salary': None,
  'tg':

In [28]:
# 1. Превращаем dict_list в DataFrame
parsed_df = pd.DataFrame(dict_list)

# 2. Объединяем с исходным df
result_df = pd.concat([df, parsed_df], axis=1)

# 3. Заменяем NaN на None (опционально)
result_df = result_df.where(pd.notna(result_df), None)

# Сохраняем или выводим
print(result_df.head())
# result_df.to_csv("output.csv", index=False)

   text_id                                               text  \
0        9  #вакансия #vacancy #Python #удаленка #flask #r...   
1       31  #ВАКАНСИЯ  #Системный_аналитик #РФ\n \nАКЦИОНЕ...   
2       28  #вакансия #vacancy #job #senior #data #DB #dat...   
3       49  #вакансия #fulltime #remote \n\n🔎 Ищем Руковод...   
4       18  #vacancy #job #analyst #travel #sirenatravel #...   

                job_title        company       salary           tg   grade  
0        Python developer      Collectly  6000-9000 $    @ann_gfio  senior  
1                    None           None         None         None    None  
2  Database Administrator  Match Systems    от 3000 $  @lex_kertis  senior  
3                    None           None         None         None    None  
4                    None           None         None         None    None  


In [29]:
result_df.to_csv('2_2_2_resume.csv', index=False)